In [1]:
!pip install langchain langgraph chromadb faiss-cpu sentence-transformers pydantic

In [3]:
!pip install -U langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)

  Attempting uninstall: langchain-text-splitters

    Found existing installation: langchain-text-splitters 0.3.11

    Uninstalling langchain-text-splitters-0.3.11:

      Successfully uninstalled langchain-text-splitters-0.3.11

   ---------- ----------------------------- 1/4 [langchain-huggingface]
  Attempting uninstall: langchain-community
   ---------- ----------------------------- 1/4 [langchain-huggingface]
    Found existing installation: langchain-community 0.3.31
   ---------- ----------------------------- 1/4 [langchain-huggingface]
   -------------------- ------------------- 2/4 [langchain-community]
   -------------------- ------------------- 2/4 [langchain-community]
    Uninstalling 

In [5]:
!pip install langgraph langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers

In [6]:
from langgraph.graph import StateGraph, END
from pydantic import BaseModel
from typing import Optional

import re
import math

from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
loader = TextLoader("data/bank-policies/loan-rules.txt")

documents = loader.load()

print(documents[0].page_content)

Loan Policy Rules

1. Minimum salary required for loan approval is ₹25,000 per month.
2. Minimum credit score must be 700.
3. Maximum Debt to Income (DTI) ratio allowed is 55%.
4. Interest rate ranges from 9% to 13% depending on credit score.
5. Maximum loan amount allowed is 20 times the monthly salary.
6. Maximum loan tenure allowed is 7 years.
7. Applicants with credit score above 750 receive lower interest rates.
8. Existing EMI obligations should not exceed 40% of monthly salary.
9. Salaried applicants must show at least 6 months of salary history.
10. Loan eligibility decreases if EMI + new EMI exceeds 60% of salary.


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

docs = splitter.split_documents(documents)

print(len(docs))

5


In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

retriever = vectorstore.as_retriever()

In [15]:
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    retriever
    | RunnableLambda(format_docs)
)

In [16]:
rag_chain.invoke("loan eligibility rules")

'Loan Policy Rules\n4. Interest rate ranges from 9% to 13% depending on credit score.\n5. Maximum loan amount allowed is 20 times the monthly salary.\n6. Maximum loan tenure allowed is 7 years.\n9. Salaried applicants must show at least 6 months of salary history.\n10. Loan eligibility decreases if EMI + new EMI exceeds 60% of salary.\n1. Minimum salary required for loan approval is ₹25,000 per month.\n2. Minimum credit score must be 700.\n3. Maximum Debt to Income (DTI) ratio allowed is 55%.'

In [17]:
class LoanState(BaseModel):

    user_input: str

    salary: Optional[int] = None
    credit_score: Optional[int] = None
    loan_amount: Optional[int] = None
    tenure_years: Optional[int] = None
    existing_emi: Optional[int] = None

    retrieved_rules: Optional[str] = None
    eligibility_status: Optional[str] = None
    approval_score: Optional[float] = None
    suggested_plan: Optional[str] = None
    final_reply: Optional[str] = None

In [ ]:
#Data collection -- Node
def collect_financials(state: LoanState):

    text = state.user_input.lower()

    salary = re.search(r'(\d+)k', text)
    if salary:
        state.salary = int(salary.group(1)) * 1000

    credit = re.search(r'credit score (\d+)', text)
    if credit:
        state.credit_score = int(credit.group(1))

    loan = re.search(r'(\d+) lak', text)
    if loan:
        state.loan_amount = int(loan.group(1)) * 100000

    tenure = re.search(r'(\d+) yr', text)
    if tenure:
        state.tenure_years = int(tenure.group(1))

    emi = re.search(r'emi (\d+)k', text)
    if emi:
        state.existing_emi = int(emi.group(1)) * 1000

    return state

In [19]:
#Elligibility check -- Node
def check_eligibility(state: LoanState):

    rules = rag_chain.invoke("loan eligibility rules")

    state.retrieved_rules = str(rules)

    if state.salary and state.credit_score:
        if state.salary >= 25000 and state.credit_score >= 700:
            state.eligibility_status = "Eligible"
        else:
            state.eligibility_status = "Not Eligible"

    return state



In [20]:
def predict_approval_chance(state: LoanState):

    score = 50

    if state.credit_score >= 750:
        score += 20
    elif state.credit_score >= 700:
        score += 10

    if state.salary >= 50000:
        score += 15

    if state.existing_emi < state.salary * 0.4:
        score += 10

    state.approval_score = min(score, 95)

    return state

In [21]:
def suggest_loan_plan(state: LoanState):

    P = state.loan_amount
    annual_rate = 0.10

    r = annual_rate / 12
    n = state.tenure_years * 12

    emi = (P * r * (1+r)**n) / ((1+r)**n - 1)

    state.suggested_plan = f"""
Loan Amount: ₹{P}
Tenure: {state.tenure_years} years
Estimated EMI: ₹{int(emi)}
"""

    return state

In [22]:
def review_response(state: LoanState):

    state.final_reply = f"""
Loan Eligibility: {state.eligibility_status}

Approval Chance: {state.approval_score}%

Recommended Loan Plan
---------------------
{state.suggested_plan}

Advice:
Maintain stable income and avoid high EMI commitments.
"""

    return state

In [23]:
workflow = StateGraph(LoanState)

workflow.add_node("collect_financials", collect_financials)
workflow.add_node("check_eligibility", check_eligibility)
workflow.add_node("predict_approval_chance", predict_approval_chance)
workflow.add_node("suggest_loan_plan", suggest_loan_plan)
workflow.add_node("review_response", review_response)

workflow.add_edge("collect_financials", "check_eligibility")
workflow.add_edge("check_eligibility", "predict_approval_chance")
workflow.add_edge("predict_approval_chance", "suggest_loan_plan")
workflow.add_edge("suggest_loan_plan", "review_response")
workflow.add_edge("review_response", END)

workflow.set_entry_point("collect_financials")

loan_app = workflow.compile()

In [24]:
query = "I earn 65k/month, credit score 740, loan required 10 lakhs for 3 yrs. Already paying emi 12k."

result = loan_app.invoke({
    "user_input": query
})

print(result["final_reply"])


Loan Eligibility: Eligible

Approval Chance: 85.0%

Recommended Loan Plan
---------------------

Loan Amount: ₹1000000
Tenure: 3 years
Estimated EMI: ₹32267


Advice:
Maintain stable income and avoid high EMI commitments.

